
# 03 — Transformer Tokenization

## IMDb Sentiment Analysis Using DistilBERT

### Objective

This notebook prepares the cleaned IMDb movie reviews for a pretrained Transformer model using the DistilBERT tokenizer.

The main objectives are to:

- Load and validate the cleaned IMDb training dataset.
- Understand subword tokenization and special tokens.
- Load a pretrained DistilBERT tokenizer.
- Explore tokenization outputs, including input IDs and attention masks.
- Analyze token sequence lengths.
- Select appropriate truncation and padding strategies.
- Tokenize and validate the training dataset.
- Save the tokenized data for subsequent model training.

**Pretrained model:** `distilbert-base-uncased`

**Input dataset:** `../data/clean/imdb_train_clean.csv`

**Expected output:** Tokenized IMDb reviews ready for Transformer fine-tuning.



## 1. Environment Setup & Data Loading

In this section, we import the required libraries and load the cleaned IMDb training dataset produced in Notebook 02.

Before tokenization, we verify the dataset structure, text column, and sentiment labels.


In [2]:

# Import libraries for file handling and data analysis
from pathlib import Path
import pandas as pd

# Improve the display of long IMDb reviews
pd.set_option("display.max_colwidth", 120)

print("Environment setup completed.")


Environment setup completed.


In [7]:

# Check whether the Transformers library is available
import importlib.util

is_installed = importlib.util.find_spec("transformers") is not None

print(f"Transformers installed: {is_installed}")


Transformers installed: True


In [3]:

# Define the path to the cleaned IMDb dataset
data_path = Path("../data/clean/imdb_train_clean.csv")

# Load the cleaned training data
train_df = pd.read_csv(data_path)

# Inspect the dataset dimensions and column names
print(f"Dataset shape: {train_df.shape}")
print(f"Columns: {train_df.columns.tolist()}")

# Display the first five reviews
train_df.head()


Dataset shape: (25000, 2)
Columns: ['text', 'label']


,text,label
0,I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first...,0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are ...",0
2,If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no coge...,0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead. The film has ...",0
4,"Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee so...",0



### 1.1 Dataset Validation

Before applying Transformer tokenization, we verify that:

- The dataset contains the expected columns (`text` and `label`).
- There are no missing or empty reviews.
- Sentiment labels contain only binary values (0 and 1).
- The dataset contains the expected 25,000 training samples.

These checks ensure that the input data is ready for tokenization.


In [4]:

# Verify the expected dataset structure
expected_columns = {"text", "label"}
assert set(train_df.columns) == expected_columns, "Unexpected dataset columns."

# Check the number of training samples
assert len(train_df) == 25_000, "Unexpected number of training samples."

# Check for missing and empty reviews
missing_reviews = train_df["text"].isna().sum()
empty_reviews = train_df["text"].fillna("").str.strip().eq("").sum()

# Verify that sentiment labels are binary
unique_labels = sorted(train_df["label"].dropna().unique().tolist())
missing_labels = train_df["label"].isna().sum()

# Display validation results
print(f"Total reviews: {len(train_df):,}")
print(f"Missing reviews: {missing_reviews}")
print(f"Empty reviews: {empty_reviews}")
print(f"Missing labels: {missing_labels}")
print(f"Unique labels: {unique_labels}")

# Stop execution if invalid data is detected
assert missing_reviews == 0, "Missing reviews detected."
assert empty_reviews == 0, "Empty reviews detected."
assert missing_labels == 0, "Missing sentiment labels detected."
assert unique_labels == [0, 1], "Invalid sentiment labels detected."

print("\nDataset validation passed.")


Total reviews: 25,000
Missing reviews: 0
Empty reviews: 0
Missing labels: 0
Unique labels: [0, 1]

Dataset validation passed.



## 2. Transformer Tokenization

DistilBERT uses WordPiece tokenization to convert text into subword tokens and numerical IDs.

In this section, we explore the tokenizer and examine the structure of model inputs.



## 3. Loading the Pretrained Tokenizer

Load the pretrained DistilBERT tokenizer and inspect its vocabulary, maximum sequence length, and special tokens.


In [6]:

from transformers import AutoTokenizer

# Load the pretrained DistilBERT tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Inspect the tokenizer configuration
print(f"Model: {model_name}")
print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Maximum sequence length: {tokenizer.model_max_length}")
print(f"Special tokens: {tokenizer.special_tokens_map}")


c:\Python-Envs\nlp-sentiment-2026\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Python-Envs\nlp-sentiment-2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kosar\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To su

Model: distilbert-base-uncased
Tokenizer: BertTokenizer
Vocabulary size: 30,522
Maximum sequence length: 512
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}



## 4. Tokenization on Sample Reviews

Apply the pretrained DistilBERT tokenizer to sample IMDb reviews and inspect the resulting tokens and token IDs.


In [8]:

# Select the first review from the cleaned IMDb dataset
sample_text = train_df.loc[0, "text"]

# Split the review into WordPiece tokens
tokens = tokenizer.tokenize(sample_text)

# Convert tokens into numerical IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# Inspect the tokenization results
print(f"Original text:\n{sample_text[:300]}...\n")
print(f"First 30 tokens:\n{tokens[:30]}\n")
print(f"First 30 token IDs:\n{token_ids[:30]}\n")
print(f"Total tokens: {len(tokens)}")


Original text:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h...

First 30 tokens:
['i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967', '.', 'i', 'also', 'heard']

First 30 token IDs:
[1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657]

Total tokens: 337


In [9]:

# Encode the sample review with special tokens
encoded_review = tokenizer(
    sample_text,
    add_special_tokens=True,
    truncation=False,
    padding=False
)

# Extract model input IDs and attention mask
input_ids = encoded_review["input_ids"]
attention_mask = encoded_review["attention_mask"]

# Convert IDs back to readable tokens
encoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Inspect the encoded review
print(f"First 15 tokens: {encoded_tokens[:15]}")
print(f"First 15 input IDs: {input_ids[:15]}")
print(f"First 15 attention mask values: {attention_mask[:15]}")
print(f"Last 5 tokens: {encoded_tokens[-5:]}")
print(f"Total input tokens: {len(input_ids)}")


First 15 tokens: ['[CLS]', 'i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all']
First 15 input IDs: [101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035]
First 15 attention mask values: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Last 5 tokens: ['of', 'a', 'plot', '.', '[SEP]']
Total input tokens: 339



## 5. Token Length Analysis

Analyze token sequence lengths across the training dataset to evaluate the impact of the model's maximum input length.


In [10]:

# Calculate token lengths for all reviews, including special tokens
train_df["token_length"] = [
    len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
    for text in train_df["text"]
]

# Summarize the token length distribution
print(train_df["token_length"].describe())

# Count reviews exceeding DistilBERT's maximum input length
max_model_length = tokenizer.model_max_length
long_reviews = (train_df["token_length"] > max_model_length).sum()

print(f"\nReviews exceeding {max_model_length} tokens: {long_reviews:,}")
print(f"Percentage: {long_reviews / len(train_df) * 100:.2f}%")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


count    25000.000000
mean       297.501640
std        221.215752
min         13.000000
25%        161.000000
50%        222.000000
75%        361.000000
max       3047.000000
Name: token_length, dtype: float64

Reviews exceeding 512 tokens: 3,319
Percentage: 13.28%


In [11]:

# Compare truncation rates for different sequence length limits
candidate_lengths = [128, 256, 384, 512]

for max_length in candidate_lengths:
    truncated_count = (train_df["token_length"] > max_length).sum()
    truncated_pct = truncated_count / len(train_df) * 100

    print(
        f"Max length {max_length:>3}: "
        f"{truncated_count:>5,} reviews truncated "
        f"({truncated_pct:.2f}%)"
    )


Max length 128: 21,975 reviews truncated (87.90%)
Max length 256: 10,281 reviews truncated (41.12%)
Max length 384: 5,659 reviews truncated (22.64%)
Max length 512: 3,319 reviews truncated (13.28%)



## 6. Truncation & Padding Strategy

Use a maximum sequence length of 512 tokens to handle long reviews. Apply truncation to sequences exceeding this limit and use dynamic padding during model training.


In [12]:

# Define the maximum input length for DistilBERT
MAX_LENGTH = 512

# Select a review longer than the model's limit
long_review = train_df.loc[
    train_df["token_length"] > MAX_LENGTH, "text"
].iloc[0]

# Tokenize the review with truncation enabled
truncated_review = tokenizer(
    long_review,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

# Compare the original and truncated lengths
original_length = len(tokenizer.encode(
    long_review,
    add_special_tokens=True,
    truncation=False
))

print(f"Original length: {original_length}")
print(f"Truncated length: {len(truncated_review['input_ids'])}")
print(f"Removed tokens: {original_length - len(truncated_review['input_ids'])}")
print(f"Last token: {tokenizer.convert_ids_to_tokens(truncated_review['input_ids'][-1])}")


Original length: 688
Truncated length: 512
Removed tokens: 176
Last token: [SEP]


In [13]:

# Select two reviews with different token lengths
sample_reviews = train_df.loc[[0, 1], "text"].tolist()

# Tokenize both reviews with dynamic padding
batch_encoding = tokenizer(
    sample_reviews,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True
)

# Inspect the padded batch
for i in range(len(sample_reviews)):
    input_ids = batch_encoding["input_ids"][i]
    attention_mask = batch_encoding["attention_mask"][i]

    print(f"Review {i + 1}:")
    print(f"  Sequence length: {len(input_ids)}")
    print(f"  Real tokens: {sum(attention_mask)}")
    print(f"  Padding tokens: {attention_mask.count(0)}")
    print()


Review 1:
  Sequence length: 339
  Real tokens: 339
  Padding tokens: 0

Review 2:
  Sequence length: 339
  Real tokens: 304
  Padding tokens: 35




## 7. Dataset Tokenization & Validation

Tokenize the cleaned IMDb training dataset using the selected maximum sequence length. Validate the resulting input IDs and attention masks before model training.


In [14]:

# Tokenize all reviews without padding
tokenized_data = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

# Inspect the tokenized dataset
print(f"Tokenized reviews: {len(tokenized_data['input_ids']):,}")
print(f"Labels: {len(train_df):,}")
print(f"Maximum sequence length: {max(map(len, tokenized_data['input_ids']))}")
print(f"Minimum sequence length: {min(map(len, tokenized_data['input_ids']))}")


Tokenized reviews: 25,000
Labels: 25,000
Maximum sequence length: 512
Minimum sequence length: 13


In [15]:

# Validate the tokenized dataset
input_ids = tokenized_data["input_ids"]
attention_masks = tokenized_data["attention_mask"]

assert len(input_ids) == len(attention_masks) == len(train_df)
assert all(1 <= len(ids) <= MAX_LENGTH for ids in input_ids)
assert all(len(ids) == len(mask) for ids, mask in zip(input_ids, attention_masks))
assert all(all(value == 1 for value in mask) for mask in attention_masks)
assert all(ids[0] == tokenizer.cls_token_id for ids in input_ids)
assert all(ids[-1] == tokenizer.sep_token_id for ids in input_ids)
assert set(train_df["label"].unique()) == {0, 1}

print("Dataset tokenization validation passed.")
print(f"Validated reviews: {len(input_ids):,}")


Dataset tokenization validation passed.
Validated reviews: 25,000



## 8. Saving Tokenized Data & Summary

Save the validated tokenized dataset and record the preprocessing configuration for reproducible model training.


In [16]:

import pickle
from pathlib import Path

# Define the output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare tokenized features and labels
processed_data = {
    "input_ids": tokenized_data["input_ids"],
    "attention_mask": tokenized_data["attention_mask"],
    "labels": train_df["label"].astype(int).tolist(),
    "model_name": model_name,
    "max_length": MAX_LENGTH,
}

# Save the tokenized dataset
output_path = output_dir / "imdb_train_tokenized.pkl"

with open(output_path, "wb") as file:
    pickle.dump(processed_data, file, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved to: {output_path}")
print(f"Total reviews: {len(processed_data['labels']):,}")


Saved to: ..\data\processed\imdb_train_tokenized.pkl
Total reviews: 25,000



### Summary

- Tokenized 25,000 cleaned IMDb reviews using the DistilBERT tokenizer.
- Selected a maximum sequence length of 512 tokens.
- Identified 3,319 reviews (13.28%) exceeding the model's input limit.
- Applied truncation and retained variable-length sequences for dynamic padding during training.
- Validated and saved the tokenized dataset for fine-tuning.
